# Panel App: Target Audit

A multi-stage application for auditing the targets within our region orthophoto.

- Upload region image
- Select samples of region
- Set parameters for CV search
    - Visualize parameters
- Run parameters on full orthophoto
    - View output
- Generate binary mask
- Audit detected targets on map
- Save targets to local geoJSON



In [37]:
# import param
# import panel as pn
# import geopandas as gpd
# import matplotlib.pyplot as plt
# from io import BytesIO
# import json
# from PIL import Image

# pn.extension('filedropper')


# class TargetAuditApp(param.Parameterized):
#     # Parameters for tracking uploaded files
#     region_image_upload = param.Parameter(default=None)
#     region_geojson_upload = param.Parameter(default=None)

#     # FileDropper widgets

#     # Accepted filetypes bug for this widget: https://github.com/holoviz/panel/issues/7153
#     # accepted_filetypes=["allowed/geojson", ".geojson"],
#     # Unable to handle our large geoTiff images
#     image_dropper = pn.widgets.FileDropper(height=100, max_file_size ="500MB", chunk_size=30000000)
#     geojson_dropper = pn.widgets.FileDropper(height=100, max_file_size ="100MB")

#     def __init__(self, **params):
#         super().__init__(**params)

#         # Link FileDropper outputs to parameters
#         self.image_dropper.param.watch(self._update_region_image, "value")
#         self.geojson_dropper.param.watch(self._update_region_geojson, "value")

#     # Update methods for parameters
#     def _update_region_image(self, event):
#         if event.new:
#             self.region_image_upload = event.new

#     def _update_region_geojson(self, event):
#         if event.new:
#             # first_file_name = list(event.new.keys())[0] # Dict of file names:bytes
#             # file_bytes_string = event.new[first_file_name].decode("utf-8") # Bytes to string
#             # self.region_geojson = json.loads(file_bytes_string)  # String to JSON dict
#             self.region_geojson_upload = event.new

#     def get_region_image(self):
#         image_upload_dict = self.region_image_upload # Stays as dict of files
#         first_file_name = list(image_upload_dict.keys())[0] # Dict of file names:bytes
#         image_stream  = BytesIO(image_upload_dict[first_file_name]) # Bytes to Stream
#         region_image = Image.open(image_stream)  # Stream to PIL image
#         return region_image

#     def get_region_geojson(self):
#         geojson_upload_dict = self.region_geojson_upload # Dict of files
#         first_file_name = list(geojson_upload_dict.keys())[0] # Dict of file names:bytes
#         file_bytes_string = geojson_upload_dict[first_file_name].decode("utf-8") # Bytes to string
#         region_geojson = json.loads(file_bytes_string)  # String to JSON dict
#         return region_geojson

#     # A method to display the uploaded region image
#     def view_image(self):
#         if self.region_image_upload:
#             try:
#                 image_data = self.get_region_image()
#                 fig, ax = plt.subplots(figsize=(4, 4))
#                 ax.imshow(image_data)
#                 ax.axis('off')
#                 return pn.pane.Matplotlib(fig)
#             except Exception as e:
#                 return f"Error displaying image: {e}"
#         else:
#             return "No image uploaded."

#     # A method to display the GeoJSON region outline
#     def view_geojson(self):
#         if self.region_geojson_upload:
#             try:
#                 region_geojson = self.get_region_geojson()
#                 return pn.pane.JSON(region_geojson, depth=2, name="Uploaded GeoJSON")
#             except Exception as e:
#                 return f"Error processing GeoJSON: {e}"
#         else:
#             return "No GeoJSON uploaded."

#     # Panel layout combining file droppers and visualizations
#     def panel(self):
#         return pn.Column(
#             pn.Row(
#                 pn.Column("**Drop Region Image Here**", self.image_dropper),
#                 pn.Column("**Drop GeoJSON Here**", self.geojson_dropper),
#             ),
#             pn.Row(
#                 pn.Column("**Uploaded Region Image**", self.view_image),
#                 pn.Column("**Uploaded GeoJSON Outline**", self.view_geojson),
#             ),
#         )


# # Run the app
# target_audit_app = TargetAuditApp()
# target_audit_app.panel().servable()

# target_audit_app.panel()

In [38]:
import sys
import geopandas as gpd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [39]:
region_crs = 32613 # Use this everywhere for consistency
visualization_crs = 4326 # Use this when we need leaflet visualizations

# region_image_path = '../input/IGNORE_Brewster-2024-all-orthophoto-UTM-32613.tif'
region_image_path = '../tile_server/input/IGNORE_reprojected_region.tif'

region_contour_geojson = '../input/interactive_proto/region_contour.geojson'
micro_routes_filename = '../input/interactive_proto/micro_routes.geojson'
targets_plants_filename = '../input/interactive_proto/targets.geojson'
depots_filename = '../input/interactive_proto/depot_points.geojson'


In [63]:
from ipyleaflet import (
    Map, GeoJSON, TileLayer, GeoData, 
    WidgetControl, LayersControl, ScaleControl, GeomanDrawControl, FullScreenControl
)
from panel.widgets import Button
import ipywidgets
import param
import panel as pn
import pandas as pd
import json
from shapely.geometry import shape

class MapView(param.Parameterized):
    # region_image_path = param.String(doc="Path to the orthophoto image")
    region_geojson_path = param.String(doc="Path to the GeoJSON file defining the region")
    targets_gdf = param.Parameter(default=None, doc="GeoPandas DF of potential targets")

    def __init__(self, **params):
        super().__init__(**params)
        self.map = None
        self.removed_targets_gdf = gpd.GeoDataFrame(columns=self.targets_gdf.columns, geometry='geometry')
        # self.sample_boxes_gdf = None
        # self.combined_gdf = None
        self.drawn_rectangles = []
        self.region_data = None
        self._initialize_region_data()
        self._initialize_map()

    def _initialize_region_data(self):
        with open(self.region_geojson_path, "r") as f:
            self.region_data = json.load(f)
        region_geometry = shape(self.region_data['features'][0]['geometry'])
        self.region_center = region_geometry.centroid

    def _initialize_map(self):
        self.map = Map(center=(self.region_center.y, self.region_center.x), zoom=16, scroll_wheel_zoom=True)

        # self.tile_layer = TileLayer(
        #     url="http://localhost:8000/{z}/{x}/{y}.png",
        #     min_zoom=15, max_zoom=22,
        #     name="Region Image"
        # )
        # self.map.add(self.tile_layer)

        button = ipywidgets.Button(description="Process Rectangles", button_type="primary")
        button.on_click(self._process_rectangles)
        self.map.add(WidgetControl(widget=button, position='bottomright'))

        self._add_region_outline_layer()
        self._add_targets_layer()
        self._add_removed_targets_layer()
        self._add_draw_control()
        self._add_map_controls()

    def _add_region_outline_layer(self):
        region_layer = GeoJSON(
            data=self.region_data, 
            style={'color': 'blue', 'fillOpacity': 0.05, 'weight': 2},
            name=self.region_data['name'])
        self.map.add(region_layer) # Add the region border to the map

    def _add_targets_layer(self):
        self.targets_layer = GeoData(
            geo_dataframe=self.targets_gdf,
            style={'color': 'black', 'radius':6, 'fillColor': 'blue', 'opacity':0.5, 'weight':1, 'fillOpacity':0.3},
            hover_style={'fillColor': 'blue' , 'fillOpacity': 0.2},
            point_style={'radius': 3, 'color': 'red', 'fillOpacity': 0.8, 'fillColor': 'blue', 'weight': 3},
            draggable=True,
            name="Identified targets"
            )
        
        def on_click_target(event, feature, properties, id):
            # Move the clicked point to the removed targets layer
            target_id = properties['target_id']
            clicked_point = self.targets_gdf[self.targets_gdf['target_id'] == target_id]
            # Remove from targets_gdf
            self.targets_gdf = self.targets_gdf[self.targets_gdf['target_id'] != target_id]
            self.targets_layer.geo_dataframe = self.targets_gdf
            # Add to removed_targets_gdf
            self.removed_targets_gdf = pd.concat([self.removed_targets_gdf, clicked_point])
            self.removed_targets_layer.geo_dataframe = self.removed_targets_gdf

        self.targets_layer.on_click(on_click_target)
        self.map.add(self.targets_layer)

    def _add_removed_targets_layer(self):
        self.removed_targets_layer = GeoData(
            geo_dataframe=self.removed_targets_gdf,
            style={'color': 'black', 'radius':6, 'fillColor': 'red', 'opacity':0.5, 'weight':1, 'fillOpacity':0.3},
            hover_style={'fillColor': 'red' , 'fillOpacity': 0.2},
            point_style={'radius': 3, 'color': 'red', 'fillOpacity': 0.8, 'fillColor': 'blue', 'weight': 3},
            draggable=True,
            name="Removed targets"
            )
        
        def on_click_removed_target(event, feature, properties, id):
            target_id = properties['target_id']
            clicked_point = self.removed_targets_gdf[self.removed_targets_gdf['target_id'] == target_id]

            self.removed_targets_gdf = self.removed_targets_gdf[self.removed_targets_gdf['target_id'] != target_id]
            self.removed_targets_layer.geo_dataframe = self.removed_targets_gdf # Remove from removed_targets_gdf
            
            self.targets_gdf = pd.concat([self.targets_gdf, clicked_point]) # Add back to targets_gdf
            self.targets_layer.geo_dataframe = self.targets_gdf

        self.removed_targets_layer.on_click(on_click_removed_target)
        self.map.add(self.removed_targets_layer)

    def _add_draw_control(self):
        self.draw_control = GeomanDrawControl()
        
        self.draw_control.circlemarker = {}
        self.draw_control.polygon = {}
        self.draw_control.polyline = {}
        self.draw_control.rectangle = {"pathOptions": {"weight": 2, "color": "green", "fillOpacity": 0.1}}
        
        self.draw_control.rotate = False
        self.draw_control.cut = False
        self.draw_control.edit = False
        self.draw_control.drag = False # Does not maintain state 
        self.draw_control.remove = False # Swap GDFs, don't remove

        self.map.add(self.draw_control)

    def _add_map_controls(self):
        self.map.add(FullScreenControl(position='topleft'))
        self.map.add(LayersControl(position="topright"))
        self.map.add(ScaleControl(position="bottomleft"))

    def _process_rectangles(self, event):
        """
        Process drawn rectangles to move points and clear rectangles.
        """
        # Check if there are any drawn geometries
        if not self.draw_control.data or not isinstance(self.draw_control.data, list):
            # print("No rectangles found or draw control data is invalid.")
            return

        # Extract valid geometries from draw control data
        drawn_geometries = [
            shape(item["geometry"])
            for item in self.draw_control.data
            if "geometry" in item
        ]

        if not drawn_geometries:
            # print("No valid rectangles were drawn.")
            return # Exit early if no valid geometries are found

        for rectangle in drawn_geometries:
            points_in_rectangle = self.targets_gdf[self.targets_gdf.geometry.within(rectangle)]
            
            # Move points to removed_targets_gdf
            self.removed_targets_gdf = pd.concat([self.removed_targets_gdf, points_in_rectangle])
            
            # Remove those points from targets_gdf
            self.targets_gdf = self.targets_gdf[~self.targets_gdf.geometry.within(rectangle)]
            
        self.targets_layer.geo_dataframe = self.targets_gdf # Update the layers
        self.removed_targets_layer.geo_dataframe = self.removed_targets_gdf
        
        self.draw_control.clear() # Clear the drawn rectangles



In [64]:
import geopandas as gpd

targets_gdf = gpd.read_file(targets_plants_filename)
targets_points_gdf = targets_gdf[['geometry','target_id']] # remove confusing cols

map_view = MapView(
    region_geojson_path = region_contour_geojson,
    targets_gdf = targets_points_gdf
)

# pn.extension()
pn.extension(design="material")

# pn.Column(map_view.map)
# pn.panel(map_view.map)
# pn.panel(map_view.map).servable()
pn.panel(map_view.map).show()


def print_latest_value(event):
    print(f"Targets count: {len(map_view.targets_layer.geo_dataframe)}")

button = pn.widgets.Button(name="Print Latest Value", button_type="primary")
button.on_click(print_latest_value)


# map_panel = pn.pane.IPyWidget(map_view.map)
# map_panel = pn.panel(map_view.map).servable();
map_panel = pn.panel(map_view.map)

# pn.template.FastListTemplate(
#     site="Panel",
#     title="Getting Started App",
#     sidebar=[button],
#     main=[map_panel],
# ).servable(); # The ; is needed in the notebook to not display the template. Its not needed in a script

Launching server at http://localhost:56606


No rectangles found or draw control data is invalid.


In [60]:
map_view.draw_control.data
print(len(map_view.draw_control.data))

2


In [49]:
print(len(map_view.targets_layer.data['features']))
print(len(map_view.targets_gdf))

1193
1193


In [43]:
# len(map_view.region_data['features'])
map_view.region_data['features']
map_view.region_data
# len(map_view.region_data['features']['geometry']['coordinates'])
len(map_view.region_data['features'][0]['geometry']['coordinates'])
map_view.region_data['features'][0]['geometry']['coordinates']

map_view.targets_layer.data

{'type': 'FeatureCollection',
 'features': [{'id': '0',
   'type': 'Feature',
   'properties': {'target_id': '4e07e911-a98a-4beb-b937-0fafc2480978'},
   'geometry': {'type': 'Point',
    'coordinates': [-103.603803918554, 30.2506277518032]}},
  {'id': '1',
   'type': 'Feature',
   'properties': {'target_id': 'af68ffec-713e-4f94-a758-5422ca6ddd4c'},
   'geometry': {'type': 'Point',
    'coordinates': [-103.60259102142537, 30.250617799616144]}},
  {'id': '2',
   'type': 'Feature',
   'properties': {'target_id': 'e22a26a0-8973-4120-b516-eea93a41576c'},
   'geometry': {'type': 'Point',
    'coordinates': [-103.60360312113374, 30.250614985269355]}},
  {'id': '3',
   'type': 'Feature',
   'properties': {'target_id': '8ae2799d-4335-4948-8441-c18ef4694aa9'},
   'geometry': {'type': 'Point',
    'coordinates': [-103.6033621837268, 30.250589951250568]}},
  {'id': '4',
   'type': 'Feature',
   'properties': {'target_id': '9c8a6bfa-f3b0-401c-a8fb-49e5e3d242b5'},
   'geometry': {'type': 'Point',
  